In [1]:
import os
import copy
from params.paths import ROOT_DIR
from file_handling.file_read_writer import read_json, write_json
from dbio.representative_db import connect_db, get_person_by_column, get_election_result_by_person_id, get_closest_person_by_name
from utils.string_process import clean_repr_name
VISUAL_DATA_DIR = os.path.join(ROOT_DIR, "data", "data_visual", "static")
SHUGIIN_REPR_LIST_DIR = os.path.join(ROOT_DIR, "data", "data_shugiin", "repr_list")
SANGIIN_REPR_LIST_DIR = os.path.join(ROOT_DIR, "data", "data_sangiin", "repr_list")
SHUGIIN_REPR_LIST_FILE_PATH = os.path.join(SHUGIIN_REPR_LIST_DIR, "20241218_repr_list.json")
SANGIIN_REPR_LIST_FILE_PATH = os.path.join(SANGIIN_REPR_LIST_DIR, "20250911_repr_list.json")

SHUGIIN_REPR_LIST = read_json(SHUGIIN_REPR_LIST_FILE_PATH)["reprs"]
SANGIIN_REPR_LIST = read_json(SANGIIN_REPR_LIST_FILE_PATH)["reprs"]

if any((not os.path.exists(PATH) for PATH in [VISUAL_DATA_DIR, SHUGIIN_REPR_LIST_FILE_PATH, SANGIIN_REPR_LIST_FILE_PATH])):
	raise ValueError("Visual data directory is not found")

print(SHUGIIN_REPR_LIST)
print(SANGIIN_REPR_LIST)


{'自民': [{'name': '逢沢  一郎君', 'yomikata': 'あいさわ  いちろう', 'kaiha': '自民', 'district': '岡山1', 'number_of_terms_lower': 13, 'number_of_terms_upper': 0}, {'name': '赤沢 亮正君', 'yomikata': 'あかざわ  りょうせい', 'kaiha': '自民', 'district': '鳥取2', 'number_of_terms_lower': 7, 'number_of_terms_upper': 0}, {'name': 'あかま 二郎君', 'yomikata': 'あかま  じろう', 'kaiha': '自民', 'district': '神奈川14', 'number_of_terms_lower': 6, 'number_of_terms_upper': 0}, {'name': '東   国幹君', 'yomikata': 'あずま  くによし', 'kaiha': '自民', 'district': '北海道6', 'number_of_terms_lower': 2, 'number_of_terms_upper': 0}, {'name': '麻生  太郎君', 'yomikata': 'あそう  たろう', 'kaiha': '自民', 'district': '福岡8', 'number_of_terms_lower': 15, 'number_of_terms_upper': 0}, {'name': 'あべ  俊子君', 'yomikata': 'あべ  としこ', 'kaiha': '自民', 'district': '（比）九州', 'number_of_terms_lower': 7, 'number_of_terms_upper': 0}, {'name': '安藤 たかお君', 'yomikata': 'あんどう  たかお', 'kaiha': '自民', 'district': '（比）東京', 'number_of_terms_lower': 2, 'number_of_terms_upper': 0}, {'name': '五十嵐  清君', 'yomikata': '

In [2]:
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Optional
from dotenv import load_dotenv

load_dotenv()	



class Representative(BaseModel):
	name:str
	kaiha:str
	district: str
	yomikata: str
	number_of_terms_lower: Optional[int] = None
	number_of_terms_upper: Optional[int] = None
	link: Optional[str] = None
	period: Optional[str] = None
	person_id: Optional[int] = None

	def to_dict(self) -> dict:
		return self.model_dump()
	
	def __str__(self) -> str:
		return f"{self.name} ({self.yomikata})"
	
	def __repr__(self) -> str:
		return self.__str__()
	
	

def iterate_repr_list(repr_list:dict) -> Representative:
	for party in repr_list.keys():
		for repr in repr_list[party]:
			yield Representative(**repr)


conn = connect_db(
    dbname="kokkaidoc",
    user="postgres",
    password=os.getenv("PSQL_DATABASE_PASSWORD"),
    host="localhost",
    port=5432,
)



In [ ]:
target_repr_list = SANGIIN_REPR_LIST
output_repr_list_with_id = []
# output_file_path = os.path.join(SHUGIIN_REPR_LIST_DIR, "20241218_repr_list_with_id.json")
output_file_path = os.path.join(SANGIIN_REPR_LIST_DIR, "20250911_repr_list_with_id.json")
with conn.cursor() as cur:

	for repr in iterate_repr_list(target_repr_list):
		repr_name_clean = clean_repr_name(repr.name)
		print("Working on ", repr.name)
		person = get_person_by_column(cur, "name_kanji", repr_name_clean)
		hiragana_person = get_person_by_column(cur, "name_kana", clean_repr_name(repr.yomikata))
		if len(person) > 1:
			print(f"{repr.name} ({repr.yomikata}) is found in multiple persons.")
			print(f"{repr.kaiha} {repr.district} {repr.number_of_terms_lower} - {repr.number_of_terms_upper}")
			for idx, p in enumerate(person):
				print(f"{idx}: {p.name_kanji} ({p.name_kana})")
				election_result = get_election_result_by_person_id(cur, p.person_id)
				print("\n".join([str(e) for e in election_result]))
		
			selected_idx = int(input(f"{repr.name} ({repr.yomikata}) is found in multiple persons. Please select the correct one: "))
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = person[int(selected_idx)].person_id
			output_repr_list_with_id.append(repr_with_id)

		elif len(person) == 1:
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = person[0].person_id
			output_repr_list_with_id.append(repr_with_id)
	
		elif len(hiragana_person) == 1:
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = hiragana_person[0].person_id
			output_repr_list_with_id.append(repr_with_id)
		else:
			candidates = get_closest_person_by_name(cur, clean_repr_name(repr.name))
			if len(candidates) == 0:
				raise ValueError(f"No person found for {repr.name}")
			for idx, candidate in enumerate(candidates):
				print(idx, candidate)
			selected_idx = int(input(f"{repr.name} ({repr.yomikata}) is not found in the database. Please select the correct one: "))
			repr_with_id = copy.deepcopy(repr)
			repr_with_id.person_id = candidates[int(selected_idx)].person.person_id
			output_repr_list_with_id.append(repr_with_id)

	
			

write_json({"reprs": [repr.to_dict() for repr in output_repr_list_with_id]}, output_file_path)


Working on  逢沢  一郎君
Working on  赤沢 亮正君
Working on  あかま 二郎君
Working on  東   国幹君
Working on  麻生  太郎君
Working on  あべ  俊子君
Working on  安藤 たかお君
Working on  五十嵐  清君
Working on  石田  真敏君
Working on  石破   茂君
Working on  石橋 林太郎君
Working on  石原  宏高君
Working on  井出  庸生君
Working on  伊藤  忠彦君
Working on  伊藤  達也君
Working on  伊東  良孝君
Working on  稲田  朋美君
Working on  井野  俊郎君
Working on  井上  信治君
Working on  井上  貴博君
Working on  井林  辰憲君
Working on  今枝 宗一郎君
Working on  岩田  和親君
Working on  岩屋   毅君
Working on  上田  英俊君
Working on  上野 賢一郎君
Working on  江渡  聡徳君
Working on  江藤   拓君
Working on  英利アルフィヤ君
Working on  遠藤  利明君
Working on  大岡  敏孝君
Working on  大串  正樹君
Working on  大空  幸星君
Working on  大西  洋平君
Working on  大野 敬太郎君
Working on  尾崎  正直君
Working on  鬼木   誠君
鬼木   誠君 (おにき  まこと) is found in multiple persons.
自民 （比）九州 5 - 0
0: 鬼木誠 (おにきまこと)
Election date: 2012-12-16 Election name: 第46回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 Result: 当選 (1)
Election date: 2014-12-14 Election name: 第47回衆議院議員総選挙 District: 福岡2区 Party: 自由民主党 R

Working on  小野寺 五典君
Working on  小渕  優子君
Working on  梶山  弘志君
Working on  勝俣  孝明君
Working on  勝目   康君
Working on  加藤  鮎子君
Working on  加藤  勝信君
Working on  加藤  竜祥君
Working on  金子  恭之君
Working on  金子  容三君
Working on  上川  陽子君
Working on  川崎 ひでと君
Working on  神田  潤一君
Working on  城内   実君
Working on  黄川田 仁志君
Working on  岸  信千世君
Working on  岸田  文雄君
Working on  木原  誠二君
Working on  木原   稔君
Working on  草間   剛君
Working on  工藤  彰三君
Working on  国定  勇人君
Working on  国光 あやの君
Working on  栗原   渉君
Working on  小池  正昭君
Working on  小泉 進次郎君
Working on  小泉  龍司君
Working on  河野  太郎君
Working on  高村  正大君
Working on  古賀   篤君
Working on  國場 幸之助君
Working on  小寺  裕雄君
Working on  後藤  茂之君
Working on  小林  茂樹君
Working on  小林  鷹之君
Working on  小林  史明君
Working on  小森  卓郎君
Working on  齋藤   健君
Working on  斎藤  洋明君
Working on  坂井   学君
Working on  坂本  哲志君
Working on  坂本 竜太郎君
Working on  笹川  博義君
Working on  佐々木  紀君
Working on  佐藤   勉君
Working on  塩崎  彰久君
Working on  柴山  昌彦君
Working on  島尻 安伊子君
Working on  島田  智明君
Working on  新谷  正義君
